# Plot dictionary-indexing results with MTEX

This notebook reads the custom `indexed.h5` output from KikuchiCUDA and plots a crystal-orientation map, an IPF-Z map, grain area, and Ni `{100}`, `{110}`, and `{111}` pole figures. It uses the saved ANG-to-UP pattern mapping and measured scan coordinates to preserve the scan geometry.

> **Orientation convention:** the HDF5 stores Bunge Euler angles in radians. Compare the plotted map with a known orientation or the original indexing reference before treating the colors as physically validated; the correct specimen/crystal reference frame depends on how the master pattern and scan were defined.

References: [MTEX Python getting started](https://mtex-toolbox.github.io/get-started), [EBSD tutorial](https://mtex-toolbox.github.io/EBSDTutorial_py.html), [grain reconstruction](https://mtex-toolbox.github.io/GrainReconstruction_py.html), [grain area and selection](https://mtex-toolbox.github.io/SelectingGrains_py.html), and [MTEX vs. Bunge convention](https://mtex-toolbox.github.io/MTEXvsBungeConvention_py.html).

## 1. Prepare a Python environment

The current MTEX Python release requires Python 3.12 or newer. In a fresh environment, install MTEX and the HDF5/plotting dependencies, then select that environment as this notebook's Jupyter kernel:

```bash
python -m pip install "mtex[all]" h5py matplotlib
python -m pip install ipykernel
python -m ipykernel install --user --name mtex-python --display-name "Python (MTEX)"
```

In Jupyter, choose **Kernel → Change Kernel → Python (MTEX)**. The Python 3.11 HPC environment used for indexing will need a Python 3.12+ environment to run the current MTEX Python package. This notebook is only a plotting notebook; it does not rerun indexing.

In [ ]:
from pathlib import Path
import json

import h5py
import matplotlib.pyplot as plt
import numpy as np
from mtex import *

# Edit this path to point to the indexed HDF5 file you downloaded.
RESULT_H5 = Path.home() / 'Downloads' / 'ni-indexed-31375306.h5'
OEM_H5 = Path.home() / 'Downloads' / 'Ni.oem'  # Optional Oxford OEM file for Ni crystal metadata

# The run used a cubic Ni master pattern. Change this if your master
# pattern was generated for a different phase/crystal symmetry.
CRYSTAL_SYMMETRY = 'm-3m'

if not RESULT_H5.is_file():
    raise FileNotFoundError(f'Update RESULT_H5; file not found: {RESULT_H5}')
print(f'Result file: {RESULT_H5}')
print(f'Ni OEM metadata file found: {OEM_H5.is_file()} ({OEM_H5})')

## 2. Inspect the result file

The output is KikuchiCUDA's own HDF5 schema, not an MTEX-native EBSD file. The next cell checks the run status and the datasets needed for map reconstruction.

In [ ]:
with h5py.File(RESULT_H5, 'r') as f:
    print('Root groups:', list(f.keys()))
    print('Root attributes:')
    for key, value in f.attrs.items():
        if key == 'metadata_json':
            try:
                run_metadata = json.loads(value.decode() if isinstance(value, bytes) else value)
                print('  metadata_json:', json.dumps(run_metadata, indent=2)[:4000])
            except Exception:
                print('  metadata_json:', value)
        else:
            print(f'  {key}: {value}')

    required = [
        'simulation_indices', 'scores', 'valid',
        'best_euler_bunge_radians',
        'ang_mapping/pattern_index', 'ang_mapping/xy_um',
        'ang_mapping/included_in_run',
    ]
    missing = [name for name in required if name not in f]
    if missing:
        raise KeyError(f'Result file is missing required datasets: {missing}')
    print('Dataset shapes:')
    for name in required:
        print(f'  {name}: {f[name].shape} {f[name].dtype}')
    if 'complete' in f.attrs and not bool(f.attrs['complete']):
        raise RuntimeError('The HDF5 file is marked incomplete; check the HPC job output.')

## 3. Rebuild the scan using the saved ANG coordinates

`pattern_index` maps each ANG row to its UP1 pattern slot. `included_in_run` marks scan rows included in this indexing run. `valid` marks indexed UP1 patterns. The code keeps all included ANG positions, labels invalid/unindexed positions as phase 0, and attaches the best NCC score as a property.

In [ ]:
with h5py.File(RESULT_H5, 'r') as f:
    pattern_index_all = np.asarray(f['ang_mapping/pattern_index'], dtype=np.int64)
    xy_all = np.asarray(f['ang_mapping/xy_um'], dtype=np.float64)
    included_all = np.asarray(f['ang_mapping/included_in_run'], dtype=bool)
    valid_up = np.asarray(f['valid'], dtype=bool)
    euler_up = np.asarray(f['best_euler_bunge_radians'], dtype=np.float64)
    scores_up = np.asarray(f['scores'], dtype=np.float32)

if xy_all.ndim != 2 or xy_all.shape[1] < 2:
    raise ValueError(f'Expected ANG xy_um with shape (n, 2+), got {xy_all.shape}')
if not (len(pattern_index_all) == len(xy_all) == len(included_all)):
    raise ValueError('ANG mapping datasets have inconsistent lengths')

ang_rows = np.flatnonzero(included_all)
up_slots = pattern_index_all[ang_rows]
if np.any(up_slots < 0) or np.any(up_slots >= len(valid_up)):
    raise IndexError('ANG mapping points outside the indexed UP pattern arrays')

xy_map = xy_all[ang_rows, :2]
map_valid = valid_up[up_slots]
map_euler = np.zeros((len(ang_rows), 3), dtype=np.float64)
map_euler[map_valid] = euler_up[up_slots[map_valid], :3]
map_score = np.full(len(ang_rows), np.nan, dtype=np.float32)
map_score[map_valid] = scores_up[up_slots[map_valid], 0]
phase_id = np.where(map_valid, 1, 0).astype(np.int32)  # 0 = not indexed; 1 = Ni

print(f'ANG positions included: {len(ang_rows):,}')
print(f'Indexed positions: {int(map_valid.sum()):,}')
print(f'Not indexed: {int((~map_valid).sum()):,}')
print(f'Scan x/y range (µm): {xy_map.min(axis=0)} to {xy_map.max(axis=0)}')

## 4. Create an MTEX EBSD object

The stored Euler angles are already in radians, so do not convert them to degrees. The optional inverse switch is deliberately off by default: only turn it on if comparison with a known orientation confirms that the stored orientation direction must be inverted to match your MTEX specimen/crystal convention.

In [ ]:
# Use the uploaded OEM's Ni crystal metadata when it is available. This file
# reports atomic number 28 (Ni), space group 225, and a = b = c = 3.56 Å.
if OEM_H5.is_file():
    with h5py.File(OEM_H5, 'r') as oem:
        ni_oem = oem['Ni']
        space_group = int(ni_oem['SpaceGroupNumber'][()])
        lattice_abc_angstrom = np.asarray(ni_oem['LatticeConstants'][:3], dtype=float)
    if space_group != 225:
        raise ValueError(f'Expected FCC Ni space group 225, got {space_group}')
    cs_ni = crystalFrame('m-3m', lattice_abc_angstrom.tolist(), mineral='Ni')
    print(f'Ni OEM: space group {space_group}, a/b/c = {lattice_abc_angstrom} Å')
else:
    # FCC Ni fallback from the OEM values above.
    cs_ni = crystalFrame(CRYSTAL_SYMMETRY, [3.56, 3.56, 3.56], mineral='Ni')
orientations = orientation.byEuler(map_euler, 'Bunge', cs_ni)

INVERT_ORIENTATIONS = False
if INVERT_ORIENTATIONS:
    orientations = inv(orientations)

# MTEX phase IDs index this list: 0 is notIndexed, 1 is Ni.
crystal_symmetries = ['notIndexed', cs_ni]
properties = {
    'best_ncc': map_score,
    'ang_row': ang_rows.astype(np.int64),
    'up_pattern_index': up_slots.astype(np.int64),
}
ebsd = EBSD(xy_map, orientations, phase_id, crystal_symmetries, properties, scanUnit='um')
ni_ebsd = ebsd['indexed']
print(ebsd)
print(f'Ni orientation count: {len(ni_ebsd):,}')

## 5. Crystal-orientation map

This colors the indexed Ni measurements by their crystal orientation using MTEX's orientation coloring.

In [ ]:
fig = plt.figure(figsize=(9, 7))
plot(ni_ebsd, ni_ebsd.orientations, micronbar='off')
plt.title('Ni crystal orientation map')
plt.tight_layout()
plt.show()

## 6. IPF-Z map

The IPF direction is the specimen Z direction. For a scan with a different reference axis, replace `zvector` with the appropriate MTEX direction.

In [ ]:
fig = plt.figure(figsize=(9, 7))
plot(ni_ebsd, ipfDirection=zvector, micronbar='off')
plt.title('Ni IPF-Z map')
plt.tight_layout()
plt.show()

## 7. Reconstruct grains and measure grain area

MTEX segments neighboring indexed pixels into grains using a misorientation threshold. The threshold and minimum grain size affect the result, so adjust them for your material and report them with any measurements. Since the scan coordinates are in µm, `grains.area` is in µm². Grains cut by the scan boundary have truncated measured areas.


In [ ]:
# Common starting settings; change these to match your grain definition.
GRAIN_BOUNDARY_ANGLE_DEG = 10.0
MIN_GRAIN_PIXELS = 5

# calcGrains writes grain IDs into its input, so pass a copy.
grains = calcGrains(
    ni_ebsd.copy(),
    angle=GRAIN_BOUNDARY_ANGLE_DEG * degree,
    minPixel=MIN_GRAIN_PIXELS,
)
grain_area_um2 = np.asarray(grains.area, dtype=np.float64)
equivalent_diameter_um = np.sqrt(4.0 * grain_area_um2 / np.pi)

print(f'Reconstructed grains: {len(grain_area_um2):,}')
print(f'Boundary threshold: {GRAIN_BOUNDARY_ANGLE_DEG:g}°')
print(f'Minimum retained grain: {MIN_GRAIN_PIXELS} pixels')
print(
    'Grain area (µm²), min / median / mean / max: '
    f'{grain_area_um2.min():.3g} / {np.median(grain_area_um2):.3g} / '
    f'{grain_area_um2.mean():.3g} / {grain_area_um2.max():.3g}'
)
print(f'Area-equivalent diameter (µm), median: {np.median(equivalent_diameter_um):.3g}')


In [ ]:
fig = plt.figure(figsize=(9, 7))
plot(grains, grain_area_um2)
plt.title('Reconstructed grain area (µm²)')
mtexColorbar(title='Area (µm²)')
# MTEX manages the axes and colorbar layout for this figure.
plt.show()


In [ ]:
# Grain area is usually strongly skewed; use a log x-axis to show its range.
fig, ax = plt.subplots(figsize=(8, 5))
positive_area = grain_area_um2[grain_area_um2 > 0]
ax.hist(positive_area, bins=40)
ax.set_xscale('log')
ax.set_xlabel('Grain area (µm²)')
ax.set_ylabel('Number of grains')
ax.set_title('Grain-area distribution')
fig.tight_layout()
plt.show()


## 8. Ni `{100}`, `{110}`, and `{111}` pole figures

The cubic-family pole figures are plotted together in one MTEX figure. `antipodal=True` treats opposite directions as equivalent, which is conventional for these diffraction pole figures.

In [ ]:
ni_poles = Miller([1, 0, 0], [1, 1, 0], [1, 1, 1], cs_ni)
plotPF(ni_ebsd.orientations, ni_poles, contourf=True, antipodal=True)
mtexColorbar()
plt.show()

## Interpretation notes

- The map positions come from `ang_mapping/xy_um`; they retain the actual scan geometry.
- Color interpretation depends on the orientation reference frame. Verify it against a known indexed grain or the original EBSD reference before using the map quantitatively.
- Pole figures use indexed points only. They show texture from the orientation solutions; they do not use NCC as a weighting factor.
- If the notebook cannot find `mtex`, confirm that the selected Jupyter kernel is the Python 3.12+ environment where you installed it.